# HeatUp 03 - analysis

Reads what is on disk and judges it - **no calculator, no calculation**. Results from notebook 02, from DFT, or from anything that writes the same files work
(last section).

| check | question | evidence |
|---|---|---|
| **mechanical** | is the stiffness tensor C positive definite (Born-Huang)? | relaxed-ion elastic tensor |
| **structural transition** | did the structure stay in its basin? | MD: block-averaged positions, cell, symmetry breaking |
| **vibrational** | soft or imaginary modes? | `cfg.vibrational.sources`: harmonic bands, quasi-harmonic volumes, MD VDOS |
| **thermodynamic** | above the T-dependent hull of its competitors? | G(T) of the selected `cfg.terms`, the same model for every phase |

The checks are independent. The ones asked for in notebook 01 are reported; MD and operating temperatures come from each `plan.json`. Verdicts are recomputed
from cached results, so changing a threshold never needs a rerun.

In [ ]:
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import heatup
from heatup import cfg, override, status, run_analysis, evaluate, evaluate_all
from heatup.analyze.report import (gate_table, style_status, show_status, status_table, show_stability_figure,
                                   plot_free_energy_terms, plot_vibrational, plot_perturbative, vibrational_table, print_layout)
from heatup.core import contributions

warnings.filterwarnings("ignore")
heatup.verbose()
if os.path.exists("heatup_project.json"):
    heatup.load_json("heatup_project.json")      # folder roots written by notebook 01
print("HeatUp", heatup.__version__, "|", cfg.phases.database_root)

## Configuration
Nothing is required: thresholds, trajectory reading, hull grid and plots are in `config.py`; the terms and the evidence follow notebook 01.

In [ ]:
TARGETS = None  # None = every phase in the database; or e.g. ["Cu3Au/L1_2"]

# Override on purpose (none of these needs a recomputation of the calculations):
# cfg.gates.enabled = ["mechanical", "thermodynamic"]   # report only these checks
# cfg.terms.vibrational = "harmonic"                    # another hull model than the one prepared; False = static hull
# cfg.vibrational.sources = ["harmonic", "quasiharmonic", "anharmonic"]

## Is everything there?
`done` is usable evidence; anything else is listed with its reason. Unfinished steps are flagged `[incomplete]` in the verdict of every check that reads them.

In [ ]:
show_status()
display(status_table(status(), problems_only=True))

## Analyse the MD runs and assess the checks
`run_analysis` runs the calculator-free analysis (`mdstats`, `vdos`, `transition`) of every finished MD run; cached.

In [ ]:
run_analysis()
dirs = [os.path.join(cfg.phases.database_root, t) for t in (TARGETS or [])] or None
reports = {f"{r['material']}/{r['label']}": r for r in
           ([evaluate(d) for d in dirs] if dirs else evaluate_all())}
display(style_status(gate_table(reports)))
display(vibrational_table(reports))
for t, r in reports.items():
    display(show_stability_figure(r))
    if (r.get("vibrational") or {}).get("available"):
        print(t, "|", r["vibrational"]["message"])
        plot_vibrational(r); plt.show()          # bands (imaginary parts in red) / quasi-harmonic volumes / MD VDOS of cfg.vibrational.sources
    for f in r["flags"]:
        print(t, f)

## Hull in detail
The consistent curve compares every phase with the same method. The overlay adds `F_VDOS - F_HA` of the target only: an *estimate* of the anharmonic stabilisation, not a hull.

In [ ]:
for t, r in reports.items():
    plot_free_energy_terms(r)
    plt.show()
    print("decomposition at operating T:", (r["thermodynamic"] or {}).get("decomposition"))

## A subset of checks / strict evidence
Only the listed checks are reported; `strict=True` turns a check with incomplete evidence into `missing`.

In [ ]:
subset = {t: evaluate(os.path.join(cfg.phases.database_root, t), gates=["mechanical", "thermodynamic"],
                      save_figure=False, strict=True) for t in reports}
for t, r in subset.items():
    print(t, "| enabled:", r["enabled"], "| skipped:", r["skipped"], "| overall:", r["overall"], "| incomplete:", r["incomplete"])

## Perturbative anharmonic diagnostics
Only if the `perturbative` step was planned (needs `phono3py`): kappa_RTA(T) and phonon linewidths from third-order force constants. A diagnostic, never a check.

In [ ]:
for t, r in reports.items():
    if r.get("perturbative"):
        plot_perturbative(r); plt.show()
        print(t, r["perturbative"])
    else:
        print(t, "no perturbative result")

## Thresholds can change without recomputing

In [ ]:
with override(**{"gates.hull_ok_eV": 0.002, "gates.zero_frac_warn": 0.002, "gates.transition_rms_A": 0.2}):
    strict = {t: evaluate(os.path.join(cfg.phases.database_root, t), save_figure=False) for t in reports}
display(style_status(gate_table(strict)))

## Results from another code (DFT, another potential)
Put the files in the folders notebook 01 created (layout: `heatup.core.layout`). Files without a HeatUp manifest are reported `external` and accepted as they are.

| evidence | file(s) | needed fields |
|---|---|---|
| structure + energy | `relaxation/CONTCAR`, `relaxation/energy.json` | `converged`, `energy_eV_per_atom`, `enthalpy_eV_per_atom`, `calculator` (a label: phases of one hull must share it) |
| mechanical | `elastic/elastic.json` | `C_GPa` (6x6, Voigt), `moduli` (`bulk_hill_GPa`, `shear_hill_GPa`, ...) |
| harmonic phonons | `phonons/harmonic/harmonic.json` | `modes` (`omega_meV`, `weight`), `imag_fraction`, `max_imag_meV`; `band` (`x`, `special_x`, `labels`, `kpoints`, `frequencies_meV`) to locate imaginary branches |
| MD | `md/<T>K/output.traj` (ASE, with momenta) + `md.json` | `temperature_K`, `frame_dt_fs` (`md.json` is written last: it marks the run complete) |